In [1]:
import os
base_path = r'C:\Users\DELL\OneDrive - Coventry University\NCIG_Research_Data'

# This will print every folder and file it finds
if os.path.exists(base_path):
    print("Folder found! Here is what is inside:")
    print(os.listdir(base_path))
else:
    print("Error: The base_path is still wrong. Please re-copy the path.")

Folder found! Here is what is inside:
['Adware-CSVs', 'Adware-CSVs.md5', 'Adware-CSVs.zip', 'Benign-CSVs', 'Benign-CSVs.md5', 'Benign-CSVs.zip', 'Ransomware-CSVs', 'Ransomware-CSVs.md5', 'Ransomware-CSVs.zip', 'Scareware-CSVs', 'Scareware-CSVs.md5', 'Scareware-CSVs.zip', 'SMSmalware-CSVs', 'SMSmalware-CSVs.md5', 'SMSmalware-CSVs.zip']


In [2]:
import pandas as pd
import os

base_path = r'C:\Users\DELL\OneDrive - Coventry University\NCIG_Research_Data'
folders = ['Benign-CSVs', 'Adware-CSVs', 'Ransomware-CSVs', 'Scareware-CSVs', 'SMSmalware-CSVs']
data_list = []

for folder in folders:
    label = 0 if 'Benign' in folder else 1
    folder_path = os.path.join(base_path, folder)
    
    # os.walk looks inside every sub-folder automatically
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                df = pd.read_csv(file_path)
                df['Label'] = label
                data_list.append(df)

if data_list:
    master_df = pd.concat(data_list, ignore_index=True)
    print(f"Verification Success: {len(data_list)} files found and merged.")
    print(f"Your Master Table is ready with shape: {master_df.shape}")
else:
    print("Still no files found. Please open one folder (e.g., Benign-CSVs) in File Explorer and tell me if you see another folder inside it.")


Verification Success: 2126 files found and merged.
Your Master Table is ready with shape: (2616579, 86)


In [3]:
import os
base_path = r'C:\Users\DELL\OneDrive - Coventry University\NCIG_Research_Data'

# This will print every folder and file it finds
if os.path.exists(base_path):
    print("Folder found! Here is what is inside:")
    print(os.listdir(base_path))
else:
    print("Error: The base_path is still wrong. Please re-copy the path.")

Folder found! Here is what is inside:
['Adware-CSVs', 'Adware-CSVs.md5', 'Adware-CSVs.zip', 'Benign-CSVs', 'Benign-CSVs.md5', 'Benign-CSVs.zip', 'Ransomware-CSVs', 'Ransomware-CSVs.md5', 'Ransomware-CSVs.zip', 'Scareware-CSVs', 'Scareware-CSVs.md5', 'Scareware-CSVs.zip', 'SMSmalware-CSVs', 'SMSmalware-CSVs.md5', 'SMSmalware-CSVs.zip']


In [4]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, confusion_matrix

In [5]:
base_path = r'C:\Users\DELL\OneDrive - Coventry University\NCIG_Research_Data'
folders = ['Benign-CSVs', 'Adware-CSVs', 'Ransomware-CSVs', 'Scareware-CSVs', 'SMSmalware-CSVs']
data_list = []

for folder in folders:
    label = 0 if 'Benign' in folder else 1
    folder_path = os.path.join(base_path, folder)
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.csv'):
                df = pd.read_csv(os.path.join(root, file))
                df['Label'] = label
                data_list.append(df)

master_df = pd.concat(data_list, ignore_index=True)
print(f"Verification Success: {len(data_list)} files found and merged.")
print(f"Master Table Shape: {master_df.shape}")

Verification Success: 2126 files found and merged.
Master Table Shape: (2616579, 86)


In [6]:
# 1. Take a sample so the kernel doesn't crash
sampled_df = master_df.sample(n=100000, random_state=42)

# 2. Clean numeric data
X = sampled_df.drop('Label', axis=1).select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan).fillna(0) 
y = sampled_df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Run simulations
models = {
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(),
    "SVM": SVC(probability=True) 
}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"--- {name} Results ---")
    print(f"F1-Score: {f1_score(y_test, preds):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, preds)}\n")

Training Random Forest...
--- Random Forest Results ---
F1-Score: 0.6412
Confusion Matrix:
[[ 7700  6146]
 [ 5631 10523]]

Training XGBoost...
--- XGBoost Results ---
F1-Score: 0.6675
Confusion Matrix:
[[ 6615  7231]
 [ 4440 11714]]

Training SVM...
--- SVM Results ---
F1-Score: 0.7000
Confusion Matrix:
[[    3 13843]
 [    0 16154]]



In [7]:
from sklearn.metrics import roc_auc_score

print("--- Final ROC-AUC Benchmarks ---")
for name, model in models.items():
    # Calculate probabilities for the ROC-AUC metric
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    print(f"{name} ROC-AUC: {auc:.4f}")


--- Final ROC-AUC Benchmarks ---
Random Forest ROC-AUC: 0.6527
XGBoost ROC-AUC: 0.6657
SVM ROC-AUC: 0.5117


In [8]:
import pandas as pd

# 1. Extract feature importance from your best-performing model (Random Forest)
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feature_names = X.columns

# 2. Create the table for your NCIG Logic
feature_importance_df = pd.DataFrame({'Behavioral_Feature': feature_names, 'Importance_Score': importances})
top_5 = feature_importance_df.sort_values(by='Importance_Score', ascending=False).head(5)

print("--- Top 5 Trigger Features for NCIG Logic ---")
print(top_5)


--- Top 5 Trigger Features for NCIG Logic ---
   Behavioral_Feature  Importance_Score
0         Source Port          0.072602
20       Flow IAT Max          0.058350
3       Flow Duration          0.057521
17     Flow Packets/s          0.057268
37      Fwd Packets/s          0.057060
